### Importing Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.metrics import classification_report

### Data Loading

In [ ]:
dataset = load_dataset('csv', data_files={
    'train': 'cleaned_train_data.csv', 
    'test': 'cleaned_test_data.csv'
})
print(dataset['train'].column_names)

### Data preprocessing

In [ ]:
# Rename columns to match expected format
dataset = dataset.rename_column("Sentence", "text")
dataset = dataset.class_encode_column("Emotion")  # converts to integers if needed
dataset = dataset.rename_column("Label", "labels")

### Tokenizer Initialization

In [ ]:
model_name = 'microsoft/deberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 128 

### Tokenization

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples['text'], 
        truncation=True, 
        padding='max_length', 
        max_length=max_length
    )

tokenized_datasets = dataset.map(preprocess_function, batched=True)

### Model Initialization

In [ ]:
num_labels = len(set(dataset['train']['labels']))
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels
)

### Metrics Setup

In [ ]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1.compute(predictions=predictions, references=labels, average='weighted')['f1'],
    }

### Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./deberta-finetuned',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    warmup_steps=500,
    max_grad_norm=1.0,
    bf16=False,
    fp16=False, # changed to False to avoid overflow error
    seed=42,
)


### Trainer Initialization

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

### Model Training

In [ ]:
trainer.train()

### Model Evaluation & Classification Report

In [ ]:
predictions_output = trainer.predict(tokenized_datasets['test'])
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

In [ ]:
target_names = [str(i) for i in range(num_labels)]

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(labels, preds, target_names=target_names))